Парсинг комментариев

Убираем дубли постов из датасета с постами

In [ ]:
import pandas as pd
df = pd.read_csv('pikabu_posts.csv')
df_new = df.drop_duplicates(subset=['post_id'], keep='first')
print(df['post_id'].nunique())
print(df_new['post_id'].nunique())
print(len(df))
print(len(df_new))
#df_new[df_new.duplicated(subset=['text'],keep=False)] #нахождение дубликатов
df_new.to_csv('pikabu_posts_clean.csv', index=False) #записываем в csv

In [3]:
import csv
import json
import os
import random
import re
import time

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# =========================================================
# 1. ФАЙЛЫ И НАСТРОЙКИ
# =========================================================

POSTS_FILE = "pikabu_posts_clean.csv"
COMMENTS_FILE = "pikabu_comments.csv"
STATE_FILE = "parsed_comment_posts.json"
FAILED_FILE = "failed_posts.json"
COMMENT_FIELDS = [
    "author",
    "author_id",
    "id",
    "indent",
    "rating",
    "text",
    "post_id"
]

PAGE_LOAD_TIMEOUT = 6
WAIT_TIMEOUT = 3

POST_DELAY_RANGE = (2, 6)
ACTION_DELAY_RANGE = (0.5, 1.2)
RETRY_DELAY_RANGE = (2, 6)

MAX_RETRIES_PER_POST = 1
MAX_EXPAND_ROUNDS = 3

In [4]:
# =========================================================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================================================

def random_sleep(delay_range):
    time.sleep(random.uniform(*delay_range))


def clean_text(text):
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def safe_int(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default

In [5]:
# =========================================================
# 3. РАБОТА С ФАЙЛАМИ
# =========================================================

def ensure_comments_csv_exists():
    if not os.path.exists(COMMENTS_FILE):
        with open(COMMENTS_FILE, "w", encoding="utf-8-sig", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=COMMENT_FIELDS)
            writer.writeheader()


def append_comments_to_csv(comments):
    if not comments:
        return

    with open(COMMENTS_FILE, "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=COMMENT_FIELDS)
        writer.writerows(comments)


def load_processed_posts():
    if not os.path.exists(STATE_FILE):
        return set()

    try:
        with open(STATE_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
            return set(data)
    except Exception:
        return set()


def save_processed_posts(processed_posts):
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(sorted(list(processed_posts)), f, ensure_ascii=False, indent=2)


def load_failed_posts():
    if not os.path.exists(FAILED_FILE):
        return []

    try:
        with open(FAILED_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return []


def save_failed_post(post_id, url, reason):
    failed_posts = load_failed_posts()

    # чтобы не дублировать один и тот же post_id много раз
    existing_ids = {item["post_id"] for item in failed_posts if "post_id" in item}
    if post_id in existing_ids:
        return

    failed_posts.append({
        "post_id": int(post_id),
        "url": url,
        "reason": str(reason)
    })

    with open(FAILED_FILE, "w", encoding="utf-8") as f:
        json.dump(failed_posts, f, ensure_ascii=False, indent=2)


In [6]:
# =========================================================
# 4. ЧТЕНИЕ ПОСТОВ
# =========================================================

def load_posts():
    df = pd.read_csv(POSTS_FILE)

    required_columns = {"post_id", "url"}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"В файле {POSTS_FILE} должны быть колонки: {required_columns}")

    posts = df[["post_id", "url"]].dropna().head(2500).to_dict(orient="records")
    return posts

In [7]:
# =========================================================
# 5. НАСТРОЙКА SELENIUM
# =========================================================

def build_driver():
    options = Options()

    # если захочешь, можешь включить headless:
    # options.add_argument("--headless=new")

    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return driver


In [8]:
# =========================================================
# 6. ДЕЙСТВИЯ НА СТРАНИЦЕ
# =========================================================

def open_post_page(driver, url):
    driver.get(url)
    random_sleep((2, 3))

    WebDriverWait(driver, WAIT_TIMEOUT).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )


def click_main_comments_button(driver):
    """
    Пытаемся раскрыть основной блок комментариев.
    Если кнопки нет - это не ошибка: возможно, комментарии уже открыты.
    """
    possible_xpaths = [
        "//button[contains(., 'Раскрыть') and contains(., 'комментар')]",
        "//button[contains(., 'Показать') and contains(., 'комментар')]",
        "//button[contains(., 'комментар')]",
        "//a[contains(., 'комментар')]"
    ]

    for xpath in possible_xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xpath)
            for button in buttons:
                if button.is_displayed() and button.is_enabled():
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
                    random_sleep(ACTION_DELAY_RANGE)
                    driver.execute_script("arguments[0].click();", button)
                    random_sleep((2, 4))
                    return True
        except Exception:
            continue

    return False


def expand_comment_threads(driver):
    """
    Раскрывает вложенные ветки комментариев.
    Несколько проходов, потому что новые кнопки могут появляться после предыдущих кликов.
    """
    already_clicked = set()
    total_clicked = 0

    for _ in range(MAX_EXPAND_ROUNDS):
        buttons = driver.execute_script("""
            const selectors = [
                'a.comment-toggle-children_collapse',
                '.comment-toggle-children_collapse',
                '[class*="toggle-children"]',
                '[class*="comment-toggle"]'
            ];

            let result = [];
            for (const selector of selectors) {
                const found = document.querySelectorAll(selector);
                for (const el of found) {
                    if (el.offsetParent !== null) {
                        result.push(el);
                    }
                }
            }
            return result;
        """)

        if not buttons:
            break

        clicked_this_round = 0

        for button in buttons:
            try:
                outer_html = button.get_attribute("outerHTML")
                if outer_html in already_clicked:
                    continue

                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
                random_sleep((0.2, 0.6))
                driver.execute_script("arguments[0].click();", button)
                already_clicked.add(outer_html)
                clicked_this_round += 1
                total_clicked += 1
                random_sleep((0.3, 0.8))
            except Exception:
                continue

        if clicked_this_round == 0:
            break

        random_sleep((1, 2))

    return total_clicked

In [9]:
# =========================================================
# 7. СБОР КОММЕНТАРИЕВ
# =========================================================

def get_all_comments_data(driver):
    comments = driver.execute_script("""
        function cleanText(text) {
            if (!text) return '';
            return text.replace(/\\s+/g, ' ').trim();
        }

        function getCommentText(commentElement) {
            const selectors = [
                '.comment__content',
                '.comment__body',
                '.comment__text',
                '[class*="comment__content"]',
                '[class*="comment__body"]',
                '[class*="comment__text"]'
            ];

            for (const selector of selectors) {
                const el = commentElement.querySelector(selector);
                if (el) {
                    const text = cleanText(el.textContent);
                    if (text) return text;
                }
            }

            const clone = commentElement.cloneNode(true);
            const garbageSelectors = [
                '.comment__rating',
                '.comment__controls',
                '.comment__header',
                '.comment__footer',
                '.comment__emotions',
                '.user__nick'
            ];

            for (const selector of garbageSelectors) {
                clone.querySelectorAll(selector).forEach(el => el.remove());
            }

            return cleanText(clone.textContent);
        }

        function getCommentAuthor(commentElement) {
            const selectors = [
                '.user__nick',
                '[class*="user__nick"]',
                '[class*="author"]'
            ];

            for (const selector of selectors) {
                const el = commentElement.querySelector(selector);
                if (el) {
                    const text = cleanText(el.textContent);
                    if (text) return text;
                }
            }
            return '';
        }

        function getCommentRating(commentElement) {
            const selectors = [
                '.comment__rating',
                '[class*="comment__rating"]',
                '[class*="rating"]'
            ];

            for (const selector of selectors) {
                const el = commentElement.querySelector(selector);
                if (el) {
                    const text = cleanText(el.textContent);
                    if (text) return text;
                }
            }
            return '';
        }

        const result = [];
        const commentElements = document.querySelectorAll('div.comment');

        commentElements.forEach(comment => {
            result.push({
                id: comment.getAttribute('data-id') || '',
                indent: comment.getAttribute('data-indent') || '0',
                author_id: comment.getAttribute('data-author-id') || '',
                author: getCommentAuthor(comment),
                rating: getCommentRating(comment),
                text: getCommentText(comment)
            });
        });

        return result;
    """)

    normalized_comments = []
    seen_ids = set()

    for comment in comments:
        comment_id = clean_text(comment.get("id", ""))
        if not comment_id:
            continue

        if comment_id in seen_ids:
            continue
        seen_ids.add(comment_id)

        normalized_comments.append({
            "author": clean_text(comment.get("author", "")),
            "author_id": clean_text(comment.get("author_id", "")),
            "id": comment_id,
            "indent": clean_text(comment.get("indent", "0")),
            "rating": clean_text(comment.get("rating", "")),
            "text": clean_text(comment.get("text", ""))
        })

    return normalized_comments

In [10]:
# =========================================================
# 8. ПАРСИНГ ОДНОГО ПОСТА
# =========================================================

def parse_one_post_comments(driver, post_id, url):
    open_post_page(driver, url)

    try:
        click_main_comments_button(driver)
    except Exception:
        pass

    random_sleep((1, 2))

    try:
        expand_comment_threads(driver)
    except Exception:
        pass

    random_sleep((1, 2))

    comments = get_all_comments_data(driver)

    for comment in comments:
        comment["post_id"] = int(post_id)

    return comments

In [11]:
# =========================================================
# 9. ОСНОВНОЙ ЦИКЛ
# =========================================================

def main():
    ensure_comments_csv_exists()

    posts = load_posts()
    processed_posts = load_processed_posts()

    print(f"Всего постов в файле: {len(posts)}")
    print(f"Уже обработано постов: {len(processed_posts)}")

    driver = build_driver()

    try:
        for idx, post in enumerate(posts, start=1):
            post_id = safe_int(post["post_id"], None)
            url = str(post["url"]).strip()

            if not post_id or not url:
                continue

            if post_id in processed_posts:
                print(f"[{idx}/{len(posts)}] post_id={post_id} уже обработан, пропускаем")
                continue

            print(f"\n[{idx}/{len(posts)}] Парсим комментарии поста {post_id}")

            success = False

            for attempt in range(1, MAX_RETRIES_PER_POST + 1):
                try:
                    print(f"Попытка {attempt}...")
                    comments = parse_one_post_comments(driver, post_id, url)

                    append_comments_to_csv(comments)

                    processed_posts.add(post_id)
                    save_processed_posts(processed_posts)

                    print(f"Сохранено комментариев: {len(comments)}")
                    success = True
                    break

                except TimeoutException as e:
                    print(f"Timeout для post_id={post_id}: {e}")
                    random_sleep(RETRY_DELAY_RANGE)

                except WebDriverException as e:
                    print(f"WebDriver ошибка для post_id={post_id}: {e}")
                    random_sleep(RETRY_DELAY_RANGE)

                except Exception as e:
                    print(f"Общая ошибка для post_id={post_id}: {e}")
                    random_sleep(RETRY_DELAY_RANGE)

            if not success:
                save_failed_post(post_id, url, "Не удалось обработать после всех попыток")
                print(f"Пост {post_id} добавлен в failed_posts.json")

            random_sleep(POST_DELAY_RANGE)

    finally:
        driver.quit()

    print("\nГотово.")


if __name__ == "__main__":
    main()

Всего постов в файле: 2500
Уже обработано постов: 2252
[1/2500] post_id=13488271 уже обработан, пропускаем
[2/2500] post_id=13483464 уже обработан, пропускаем
[3/2500] post_id=13352593 уже обработан, пропускаем
[4/2500] post_id=13347928 уже обработан, пропускаем
[5/2500] post_id=13344925 уже обработан, пропускаем
[6/2500] post_id=13308850 уже обработан, пропускаем
[7/2500] post_id=13215337 уже обработан, пропускаем
[8/2500] post_id=13212542 уже обработан, пропускаем
[9/2500] post_id=13207241 уже обработан, пропускаем
[10/2500] post_id=13158993 уже обработан, пропускаем
[11/2500] post_id=13154939 уже обработан, пропускаем
[12/2500] post_id=13148492 уже обработан, пропускаем
[13/2500] post_id=13034332 уже обработан, пропускаем
[14/2500] post_id=13027907 уже обработан, пропускаем
[15/2500] post_id=13000240 уже обработан, пропускаем
[16/2500] post_id=12990249 уже обработан, пропускаем
[17/2500] post_id=12927426 уже обработан, пропускаем
[18/2500] post_id=12910106 уже обработан, пропускаем
